# Assignment 4 — Deep Reinforcement Learning on CartPole
### Walkthrough & Workspace Notebook

This notebook guides you through the assignment in four progressive parts:

| Part | Status | File(s) |
|------|--------|---------|
| **A – Shared Networks** | 🟡 Your task | `models.py` |
| **B – REINFORCE** | ✅ Provided baseline — read & run | `agents/pg_agent.py` |
| **C – DQN** | 🔴 Your task | `agents/dqn_agent.py`, `models.py` (`QNet`) |
| **D – Actor-Critic** | 🔴 Your task | `agents/ac_agent.py`, `models.py` (`CriticNet`) |

**How to use this notebook:**

1. Run every cell in order — the REINFORCE section is fully runnable and will display four diagnostic plots.
2. Read the REINFORCE code carefully before implementing DQN and Actor-Critic.
3. Fill in the `# TODO` sections and use the smoke-test and comparison cells at the end to validate your work.
4. Detailed grading criteria are in `ASSIGNMENT.md`.

> **Run from the project root** — this notebook uses relative imports (`from models import …`).  
> The first setup cell adds the project root to `sys.path` automatically.


In [3]:
# Dependency bootstrap — equivalent to: pip install -r requirements.txt
# Safe to rerun at any time; automatically falls back to the official PyPI index.
import subprocess, sys

def _pip_install(extra_args=()):
    """Run pip install -r requirements.txt with given extra index args."""
    cmd = [
        sys.executable, "-m", "pip", "install",
        "-r", "requirements.txt",
        "--prefer-binary",
        "--disable-pip-version-check",
        "--no-input",
    ] + list(extra_args)
    proc = subprocess.run(cmd)
    return proc.returncode

print("Installing dependencies (trying Aliyun mirror first)…")
rc = _pip_install([
    "--index-url", "https://mirrors.aliyun.com/pypi/simple",
])

if rc != 0:
    print(f"Mirror returned exit code {rc} — retrying against official PyPI…")
    rc = _pip_install()
    if rc != 0:
        raise RuntimeError(
            f"pip install -r requirements.txt failed with exit code {rc}. "
            "Check requirements.txt and your network connection."
        )

print("\u2713 All dependencies installed successfully.")


Installing dependencies (trying Aliyun mirror first)…
Looking in indexes: https://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 1.3 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 1.5 MB/s  0:00:19m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.5 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [moviepy]m6/7 [moviepy]m]mpeg]
✓ All dependencies installed successfully.


In [4]:
# ── Setup: make relative imports work wherever JupyterLab opens this notebook ──
import sys, os
from pathlib import Path

# Keep headless classroom rendering quiet and independent of host devices.
runtime_dir = Path("/tmp") / f"aicosmos-runtime-{os.getuid()}"
runtime_dir.mkdir(mode=0o700, parents=True, exist_ok=True)
runtime_dir.chmod(0o700)
os.environ.setdefault("XDG_RUNTIME_DIR", str(runtime_dir))
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")

# The project root is the directory that contains this notebook.
PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)          # so run.py and the agents can find each other

# Quick sanity check
required = [
    "models.py", "trainer.py", "run.py", "replay_buffer.py",
    "agents/base.py", "agents/pg_agent.py",
    "agents/dqn_agent.py", "agents/ac_agent.py",
]
missing = [r for r in required if not (PROJECT_ROOT / r).exists()]
if missing:
    print("WARNING – missing files:", missing)
else:
    print("All project files found ✓")
print("Python:", sys.version.split()[0], "  |  Project root:", PROJECT_ROOT)


All project files found ✓
Python: 3.11.6   |  Project root: /home/jovyan/work/aicosmos/pt8hqh8frjsi7fwc


In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["figure.dpi"] = 110

print("torch  :", torch.__version__)
print("gym    :", gym.__version__)
print("numpy  :", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("CUDA available:", torch.cuda.is_available())


torch  : 2.7.1+cu128
gym    : 1.3.0
numpy  : 2.3.5
matplotlib: 3.9.0
CUDA available: True


---
## 1 · Environment & State Space

`CartPole-v1` terminates when the pole angle exceeds ±12° or the cart moves ±2.4 units from centre, or after 500 steps.  
The reward is +1 for every step the pole stays up. A perfect episode scores **500**.

| Index | Observation | Min | Max |
|-------|-------------|-----|-----|
| 0 | Cart position | −4.8 | 4.8 |
| 1 | Cart velocity | −∞ | ∞ |
| 2 | Pole angle (rad) | −0.418 | 0.418 |
| 3 | Pole angular velocity | −∞ | ∞ |

Actions: **0** = push left, **1** = push right.


In [6]:
env = gym.make("CartPole-v1")
obs, _ = env.reset(seed=42)
print("Observation space :", env.observation_space)
print("Action space      :", env.action_space)
print("Sample observation:", obs.round(4))

# Run one random episode to see the shape of raw returns
total = 0
terminated = truncated = False
while not terminated and not truncated:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    total += reward
print(f"Random-policy episode return: {total:.0f}  (expect ~15–30)")
env.close()


Observation space : Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Action space      : Discrete(2)
Sample observation: [ 0.0274 -0.0061  0.0359  0.0197]
Random-policy episode return: 37  (expect ~15–30)


---
## 2 · Part A — Shared Networks  *(your task — `models.py`)*

Open **`models.py`** and implement the three network classes before running the next cell.

```
PolicyNet(obs_dim, act_dim, hidden=128)  →  logits  (batch, act_dim)
QNet(obs_dim, act_dim, hidden=128)       →  Q-values (batch, act_dim)   ← Part C
CriticNet(obs_dim, hidden=128)           →  V-value  (batch, 1)          ← Part D
```

`PolicyNet` is already provided as a reference. Implement `QNet` and `CriticNet` with the same two-layer MLP pattern.


In [7]:
from models import PolicyNet, QNet, CriticNet
import torch

obs_dim, act_dim = 4, 2
x = torch.randn(8, obs_dim)      # batch of 8 states

# ---- PolicyNet (provided) ----
policy_net = PolicyNet(obs_dim, act_dim)
logits = policy_net(x)
print("PolicyNet output shape:", logits.shape)   # expect (8, 2)
assert logits.shape == (8, act_dim), f"Expected (8,{act_dim}), got {logits.shape}"
print("PolicyNet ✓")

# ---- QNet (TODO Part C) ----
try:
    qnet = QNet(obs_dim, act_dim)
    q_vals = qnet(x)
    print("QNet output shape:", q_vals.shape)     # expect (8, 2)
    assert q_vals.shape == (8, act_dim)
    print("QNet ✓")
except Exception as e:
    print("QNet not yet implemented:", e)

# ---- CriticNet (TODO Part D) ----
try:
    critic = CriticNet(obs_dim)
    v_vals = critic(x)
    print("CriticNet output shape:", v_vals.shape)  # expect (8, 1)
    assert v_vals.shape == (8, 1)
    print("CriticNet ✓")
except Exception as e:
    print("CriticNet not yet implemented:", e)


PolicyNet output shape: torch.Size([8, 2])
PolicyNet ✓
QNet not yet implemented: QNet.__init__() takes 1 positional argument but 3 were given
CriticNet not yet implemented: CriticNet.__init__() takes 1 positional argument but 2 were given


---
## 3 · Part B — REINFORCE Baseline  *(fully implemented — study this carefully)*

REINFORCE is an **on-policy, Monte-Carlo** policy-gradient algorithm.  
After every complete episode it computes the discounted return $G_t$ and ascends:

$$\nabla_\theta J(\theta) \approx \sum_{t=0}^{T} G_t \, \nabla_\theta \log \pi_\theta(a_t \mid s_t)$$

Key design choices already coded in `agents/pg_agent.py`:

* **Return normalisation** — subtract mean, divide by std — reduces gradient variance.
* **Gradient clipping** (`max_norm=1.0`) — prevents explosive updates early in training.
* **Greedy evaluation** — `argmax` over logits, no sampling, for fair comparison.

The four cells below train a fresh `PGAgent`, display the four required diagnostic plots,  
and save a checkpoint to `output/pg/`.  Training 400 episodes takes ~60–90 s on CPU.


In [8]:
# ── REINFORCE training  (≈ 60-90 s on CPU) ────────────────────────────
import torch
from agents.pg_agent import PGAgent

SEED       = 42
N_EPISODES = 400
GAMMA      = 0.99
LR         = 5e-4

torch.manual_seed(SEED)
# CartPole is small; CPU is the classroom-safe default across all images.
# Set USE_CUDA=True only when you explicitly want GPU acceleration.
USE_CUDA = False
device = torch.device("cuda:0" if USE_CUDA and torch.cuda.is_available() else "cpu")
print(f"Training on {device} …")

agent = PGAgent(
    config={"gamma": GAMMA, "lr": LR, "hidden": 128},
    env_name="CartPole-v1",
    device=device,
)

# Always derive input placement from the live model, not from a stale global.
policy_device = next(agent.policy.parameters()).device
def policy_tensor(value):
    return torch.as_tensor(value, dtype=torch.float32, device=policy_device)

reward_history = []
loss_history   = []
action_probs_history = []   # for policy-confidence plot
state_traj_history   = []   # for state-trajectory plot

env_vis = gym.make("CartPole-v1")   # separate env for state recording

for ep in range(N_EPISODES):
    # ---------- standard train step ----------
    state, _ = agent.env.reset(seed=SEED + ep)
    state_t = policy_tensor(state)
    metrics = agent.train(state_t, episode=ep)
    reward_history.append(metrics["reward"])
    loss_history.append(metrics["loss"])

    # ---------- record action probs (greedy, no grad) ----------
    with torch.no_grad():
        test_state = policy_tensor(env_vis.reset(seed=0)[0])
        logits = agent.policy(test_state)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    action_probs_history.append(probs)

    # ---------- record one short trajectory every 50 eps ----------
    if ep % 50 == 0:
        traj = []
        s, _ = env_vis.reset(seed=0)
        done = False
        while not done:
            with torch.no_grad():
                a = int(torch.argmax(agent.policy(
                    policy_tensor(s))).item())
            s, _, term, trunc, _ = env_vis.step(a)
            traj.append(s.copy())
            done = term or trunc
        state_traj_history.append((ep, traj))

    if (ep + 1) % 50 == 0:
        avg = np.mean(reward_history[-50:])
        print(f"  Episode {ep+1:>4}  avg-50={avg:>6.1f}  loss={metrics['loss']:>8.3f}")

env_vis.close()
print("\nTraining complete.")

# Save checkpoint
import os; os.makedirs("output/pg", exist_ok=True)
agent.save("output/pg")
print("Checkpoint saved → output/pg/policy.pt")


Training on cuda …


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)

### Plot 1 · Learning Curve

The raw episode return (grey) fluctuates a lot because REINFORCE is a high-variance estimator.  
The moving average (blue) shows the genuine trend — look for it crossing ~200 around episode 200–300.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: rewards ──
ax = axes[0]
window = 20
raw   = np.array(reward_history)
ma    = np.convolve(raw, np.ones(window)/window, mode="valid")
ax.plot(raw,  color="lightsteelblue", alpha=0.5, linewidth=0.8, label="Episode return")
ax.plot(np.arange(window-1, len(raw)), ma,
        color="steelblue", linewidth=2, label=f"{window}-ep moving avg")
ax.axhline(500, color="green", linestyle="--", linewidth=1, label="Max (500)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("REINFORCE Learning Curve")
ax.legend(fontsize=8)
ax.set_ylim(0, 530)

# ── Right: policy loss ──
ax2 = axes[1]
ax2.plot(loss_history, color="tomato", linewidth=0.8, alpha=0.7)
window2 = 20
loss_ma = np.convolve(loss_history, np.ones(window2)/window2, mode="valid")
ax2.plot(np.arange(window2-1, len(loss_history)), loss_ma,
         color="darkred", linewidth=2, label=f"{window2}-ep moving avg")
ax2.set_xlabel("Episode")
ax2.set_ylabel("Policy loss")
ax2.set_title("REINFORCE Policy Loss")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("output/pg/learning_curve.png", bbox_inches="tight")
plt.show()
print("Saved → output/pg/learning_curve.png")


### Plot 2 · Policy Confidence

At training start the policy is nearly uniform (∼50 / 50).  
As learning progresses the policy becomes more decisive — one action dominates for the canonical start state.  
This indicates the policy has genuinely learned and is not just getting lucky.


In [ ]:
probs_arr = np.array(action_probs_history)   # shape (N_EPISODES, 2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(probs_arr[:, 0], label="P(left)", color="steelblue",  linewidth=1.5)
ax.plot(probs_arr[:, 1], label="P(right)", color="tomato", linewidth=1.5)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.6, label="Uniform")
ax.set_xlabel("Episode")
ax.set_ylabel("Action probability")
ax.set_title("Policy Confidence on Fixed Start State")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("output/pg/policy_confidence.png", bbox_inches="tight")
plt.show()
print("Saved → output/pg/policy_confidence.png")


### Plot 3 · State Trajectories Over Training

Each line shows the **pole angle** (obs index 2) over one greedy episode, sampled every 50 training episodes.  
Early trajectories diverge quickly (short episodes) — later trajectories oscillate near zero as the policy learns to balance.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

cmap = plt.cm.viridis
n = len(state_traj_history)
for i, (ep, traj) in enumerate(state_traj_history):
    angles = [s[2] for s in traj]         # pole angle (index 2)
    color  = cmap(i / max(n - 1, 1))
    ax.plot(angles, color=color, alpha=0.8, linewidth=1.3, label=f"ep {ep}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(
    vmin=state_traj_history[0][0], vmax=state_traj_history[-1][0]))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.01)
cbar.set_label("Training episode")

ax.axhline(0,  color="black", linewidth=0.8, linestyle="--")
ax.axhline( 0.2094, color="red",  linewidth=1, linestyle=":", label="Termination ±12°")
ax.axhline(-0.2094, color="red",  linewidth=1, linestyle=":")
ax.set_xlabel("Step within episode")
ax.set_ylabel("Pole angle (rad)")
ax.set_title("State Trajectories — Pole Angle Over Training")
handles, labels = ax.get_legend_handles_labels()
# keep only the ep labels + termination label
unique = dict(zip(labels, handles))
ax.legend(list(unique.values()), list(unique.keys()), fontsize=7, loc="upper right", ncol=2)
plt.tight_layout()
plt.savefig("output/pg/state_trajectories.png", bbox_inches="tight")
plt.show()
print("Saved → output/pg/state_trajectories.png")


### Plot 4 · Rendered Environment Frame

A single rendered frame from the *final* trained policy's greedy rollout.  
The cart should be near the centre and the pole nearly vertical — the policy has learned to balance.


In [ ]:
# Render a single frame from a greedy rollout of the trained policy
env_render = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = env_render.reset(seed=0)

frames = []
terminated = truncated = False
while not terminated and not truncated:
    frames.append(env_render.render())
    with torch.no_grad():
        a = int(torch.argmax(
            agent.policy(policy_tensor(obs))).item())
    obs, _, terminated, truncated, _ = env_render.step(a)
env_render.close()

mid_frame = frames[len(frames) // 2]   # pick the midpoint for a balanced pose

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.imshow(mid_frame)
ax.axis("off")
ax.set_title(f"Trained REINFORCE policy — episode length {len(frames)} steps")
plt.tight_layout()
plt.savefig("output/pg/rendered_frame.png", bbox_inches="tight")
plt.show()
print(f"Episode lasted {len(frames)} steps  (max 500)")
print("Saved → output/pg/rendered_frame.png")


---
## 4 · Part C — DQN  *(your task)*

### What you must implement

**`models.py` → `QNet`**

The Q-network maps a state vector `(batch, obs_dim)` to Q-values `(batch, act_dim)`.  
Use the same two-layer MLP pattern as `PolicyNet`.

**`agents/dqn_agent.py` → `DQNAgent`**

Open `agents/dqn_agent.py` and complete every `TODO` marker:

```python
# TODO (Part C — __init__)
#   Create self.qnet and self.target_qnet (copy of qnet, requires_grad=False)
#   Create self.optimizer (Adam on qnet.parameters())

# TODO (Part C — select_action)
#   With probability self.epsilon return a random action (exploration)
#   Otherwise return argmax of self.qnet(state)

# TODO (Part C — store)
#   Call self.replay.add(state, action, reward, next_state, terminated)

# TODO (Part C — update)
#   Skip if len(self.replay) < self.batch_size
#   Sample a minibatch
#   Compute Bellman target: y = r + gamma * (1 - done) * max Q_target(s')
#   Compute MSE loss between Q(s,a) and y.detach()
#   Zero grad, backward, step
#   Return {"loss": loss_val}

# TODO (Part C — synchronize)
#   Copy qnet weights into target_qnet

# TODO (Part C — save / load)
#   torch.save / torch.load self.qnet.state_dict()
```

### Key concepts to review

| Concept | Why it matters |
|---------|---------------|
| **Replay buffer** | Breaks temporal correlation in transitions |
| **Target network** | Stabilises the Bellman regression target |
| **Epsilon-greedy** | Balances exploration vs exploitation |
| **Terminal masking** | Prevents bootstrapping past episode end |

After implementing, run the smoke test below.


In [ ]:
# ── Part C smoke test — uncomment after implementing DQNAgent ──────────────
# import importlib, agents.dqn_agent as dqn_mod
# importlib.reload(dqn_mod)
# from agents.dqn_agent import DQNAgent
#
# dqn = DQNAgent(
#     config={"lr": 5e-4, "epsilon": 0.1, "buffer_size": 500, "batch_size": 32},
#     env_name="CartPole-v1",
# )
# state, _ = dqn.env.reset()
# state_t = torch.as_tensor(state, dtype=torch.float32, device=dqn.device)
# for _ in range(5):
#     metrics = dqn.train(state_t, episode=0)
#     state_t = torch.as_tensor(dqn.env.reset()[0], dtype=torch.float32, device=dqn.device)
# print("DQN smoke test passed — reward:", metrics["reward"])


---
## 5 · Part D — One-Step Actor-Critic  *(your task)*

### What you must implement

**`models.py` → `CriticNet`**

The critic maps a state vector `(batch, obs_dim)` to a scalar value estimate `(batch, 1)`.  
Use the same two-layer MLP pattern, but end with a single linear output unit (no softmax).

**`agents/ac_agent.py` → `ACAgent`**

Open `agents/ac_agent.py` and complete every `TODO` marker:

```python
# TODO (Part D — __init__)
#   Create self.actor (PolicyNet) and self.actor_optimizer
#   Create self.critic (CriticNet) and self.critic_optimizer

# TODO (Part D — select_action)
#   Sample from Categorical(logits=self.actor(state))
#   Return action.item(), {"log_prob": log_prob}

# TODO (Part D — store)
#   Store (reward, log_prob, state, next_state, terminated) for update()

# TODO (Part D — update)
#   Compute one-step TD target:  y = r + gamma * (1-done) * V(s')
#   Compute TD error:  delta = y.detach() - V(s)
#   Critic loss = delta^2   (MSE)
#   Actor loss  = -log_prob * delta.detach()
#   Update critic then actor
#   Clear buffers; return losses

# TODO (Part D — save / load)
#   Save/load both actor and critic state dicts
```

### Key concept — advantage estimate

The TD error $\delta_t = R_{t+1} + \gamma V_w(S_{t+1}) - V_w(S_t)$ acts as a  
**low-variance advantage estimate** that tells the actor whether the action it took  
was better or worse than the critic's expectation.  Crucially, the **advantage must be  
detached** from the critic's computation graph when used to update the actor.

After implementing, run the smoke test below.


In [ ]:
# ── Part D smoke test — uncomment after implementing ACAgent ────────────────
# import importlib, agents.ac_agent as ac_mod
# importlib.reload(ac_mod)
# from agents.ac_agent import ACAgent
#
# ac = ACAgent(
#     config={"lr": 5e-4, "critic_lr": 5e-4},
#     env_name="CartPole-v1",
# )
# state, _ = ac.env.reset()
# state_t = torch.as_tensor(state, dtype=torch.float32, device=ac.device)
# for _ in range(5):
#     metrics = ac.train(state_t, episode=0)
#     state_t = torch.as_tensor(ac.env.reset()[0], dtype=torch.float32, device=ac.device)
# print("AC smoke test passed — reward:", metrics["reward"])


---
## 6 · Controlled Comparison  *(run after all three agents pass their smoke tests)*

Use a **shared episode budget** (≥500) and **at least three random seeds**.  
The `run.py` CLI handles this for you:

```bash
python run.py --cmd compare --episodes 500 --out_dir output/final_comparison
```

Or call the `Trainer` directly from this notebook:


In [ ]:
# ── Final comparison — uncomment after all three agents are implemented ────
# from trainer import Trainer
#
# HYPERPARAMS = {
#     "pg":  {"lr": 5e-4, "gamma": 0.99},
#     "dqn": {"lr": 5e-4, "epsilon": 0.1, "buffer_size": 10000, "batch_size": 32},
#     "ac":  {"lr": 5e-4, "critic_lr": 5e-4, "gamma": 0.99},
# }
# trainer = Trainer(hyperparams_map=HYPERPARAMS)
# results = trainer.compare(
#     agent_keys=["pg", "dqn", "ac"],
#     env_name="CartPole-v1",
#     out_dir="output/final_comparison",
#     num_train_episodes=500,
#     num_eval_episodes=5,
# )


---
## 7 · Analysis Prompts

Answer these in a markdown cell below after you have all three agents running.

1. Which algorithm starts improving first?  What property of each update rule explains this?
2. How does replay break the temporal correlation that hurts plain SGD on sequences?
3. What happens to DQN stability when you synchronise the target network too frequently?  Too rarely?
4. Compare the variance of REINFORCE returns (from the learning curve above) with Actor-Critic returns.  
   What mechanism in AC reduces variance?
5. Which findings from CartPole are likely to generalise to harder environments, and which are specific to this task?


*(Double-click this cell and write your analysis here.)*


---
## 8 · Submission Checklist

- [ ] `PolicyNet`, `QNet`, `CriticNet` are implemented; shapes verified in §2.
- [ ] REINFORCE smoke test (§3) runs without errors; four plots are visible.
- [ ] DQN smoke test (§4) runs without errors.
- [ ] Actor-Critic smoke test (§5) runs without errors.
- [ ] Final comparison (§6) runs across ≥ 3 seeds; `output/final_comparison/compare_rewards.png` saved.
- [ ] Analysis questions (§7) are answered.
- [ ] This notebook is fully executed top-to-bottom (Kernel → Restart & Run All).
- [ ] `output/` folder with figures and checkpoints is included in submission.
- [ ] Virtual environments, `.pyc` files, and large intermediate checkpoints are excluded.
